In [1]:
import requests

session = requests.Session()

session.post('https://www.space-track.org/ajaxauth/login', data={
    'identity': 'dejuakim@gmail.com',
    'password': 'siawevengersaiffel'
})

url = (
    "https://www.space-track.org/basicspacedata/query"
    "/class/gp"
    "/OBJECT_TYPE/PAYLOAD"
    "/MEAN_MOTION/13.7--15.8"
    "/ECCENTRICITY/<0.25"
    "/INCLINATION/>40"
    "/EPOCH/>now-30"
    "/orderby/OBJECT_NAME"
    "/format/json"
)

response = session.get(url)
tle_list = response.json()

print(f"수신된 위성 수: {len(tle_list)}")
print("\n위성 이름 샘플 (처음 30개):")
for sat in tle_list[:30]:
    print(f"  {sat['OBJECT_NAME']:<30} 고도추정: MEAN_MOTION={sat['MEAN_MOTION']}")

수신된 위성 수: 13555

위성 이름 샘플 (처음 30개):
  3CAT-4                         고도추정: MEAN_MOTION=15.02083385
  3CAT-5/A (TYVAK-0161)          고도추정: MEAN_MOTION=15.45123878
  3CAT-5/B (TYVAK-0162)          고도추정: MEAN_MOTION=15.46417224
  6GSTARLAB                      고도추정: MEAN_MOTION=15.18216072
  A-SEANSAT-PG1                  고도추정: MEAN_MOTION=15.35854509
  AAC-AIS-SAT-1                  고도추정: MEAN_MOTION=14.91830512
  AAC-AIS-SAT2                   고도추정: MEAN_MOTION=14.85301521
  AAC-AIS-SAT3                   고도추정: MEAN_MOTION=14.84553216
  AAC-HSI-SAT1                   고도추정: MEAN_MOTION=15.12407078
  AAC-HSI-SAT2                   고도추정: MEAN_MOTION=15.20160301
  AAC-HSI-SAT3                   고도추정: MEAN_MOTION=15.15613916
  AAC-IO1                        고도추정: MEAN_MOTION=14.91374413
  AAU CUBESAT                    고도추정: MEAN_MOTION=14.23945655
  AAUSAT3                        고도추정: MEAN_MOTION=14.39899529
  AC1-001                        고도추정: MEAN_MOTION=15.18202622
  AC1-002          

In [2]:
# OBJECT_NAME 기반이 아닌
# RCS_SIZE 기준 추가 (위성 크기)
# 지구관측 위성은 보통 MEDIUM 또는 LARGE

url = (
    "https://www.space-track.org/basicspacedata/query"
    "/class/gp"
    "/OBJECT_TYPE/PAYLOAD"
    "/MEAN_MOTION/13.7--15.8"
    "/ECCENTRICITY/<0.25"
    "/INCLINATION/>40"
    "/RCS_SIZE/MEDIUM,LARGE"   # 추가
    "/EPOCH/>now-30"
    "/orderby/OBJECT_NAME"
    "/format/json"
)

response = session.get(url)
tle_list_filtered = response.json()
print(f"RCS 필터 후 위성 수: {len(tle_list_filtered)}")

RCS 필터 후 위성 수: 10378


In [3]:
import requests

response = requests.get(
    "https://celestrak.org/SOCRATES/query.php",
    params={"GROUP": "earth-obs", "FORMAT": "tle"}
)
print(f"상태코드: {response.status_code}")
print(response.text[:500])

상태코드: 404
<!DOCTYPE html>
<html>
<head>
<meta http-equiv="Content-Type" content="text/html; charset=iso-8859-1"/>
<title>404 - File Not Found.</title>
<style type="text/css">
body{margin:0;font-size:.7em;font-family:Verdana, Arial, Helvetica, sans-serif;background:#EEEEEE;}
fieldset{padding:0 15px 10px 15px;}
h1{font-size:2.4em;margin:0;color:#FFF;}
h2{font-size:1.7em;margin:0;color:#CC0000;}
h3{font-size:1.2em;margin:10px 0 0 0;color:#000000;}
#header{width:96%;margin:0 0 0 0;padding:6px 2% 6p


In [4]:
import requests

# CelesTrak 새 URL 형식
urls_to_try = [
    "https://celestrak.org/SOCRATES/query.php?GROUP=earth-obs&FORMAT=tle",
    "https://celestrak.org/pub/TLE/catalog.txt",
    "https://celestrak.org/SOCRATES/",
    "https://celestrak.org/pub/TLE/eo-ops.txt",
    "https://celestrak.org/pub/TLE/resource.txt",
]

for url in urls_to_try:
    r = requests.get(url, timeout=10)
    print(f"{r.status_code} | {url}")

404 | https://celestrak.org/SOCRATES/query.php?GROUP=earth-obs&FORMAT=tle
403 | https://celestrak.org/pub/TLE/catalog.txt
200 | https://celestrak.org/SOCRATES/
403 | https://celestrak.org/pub/TLE/eo-ops.txt
403 | https://celestrak.org/pub/TLE/resource.txt


In [5]:
# CelesTrak GP 데이터 API (새 형식)
urls_to_try = [
    "https://celestrak.org/SOCRATES/query.php?GROUP=earth-obs&FORMAT=json",
    "https://celestrak.org/cgi-bin/TLE.cgi?GROUP=earth-obs&FORMAT=tle",
    "https://celestrak.org/SOCRATES/query.php?CATNR=25544&FORMAT=tle",  # ISS 테스트
    "https://celestrak.org/pub/TLE/active.txt",
    "https://celestrak.org/satcat/records.csv",
]

for url in urls_to_try:
    try:
        r = requests.get(url, timeout=10)
        print(f"{r.status_code} | {url}")
    except Exception as e:
        print(f"ERROR | {url} | {e}")

404 | https://celestrak.org/SOCRATES/query.php?GROUP=earth-obs&FORMAT=json
404 | https://celestrak.org/cgi-bin/TLE.cgi?GROUP=earth-obs&FORMAT=tle
404 | https://celestrak.org/SOCRATES/query.php?CATNR=25544&FORMAT=tle
ERROR | https://celestrak.org/pub/TLE/active.txt | HTTPSConnectionPool(host='celestrak.org', port=443): Max retries exceeded with url: /pub/TLE/active.txt (Caused by ConnectTimeoutError(<HTTPSConnection(host='celestrak.org', port=443) at 0x24276eecad0>, 'Connection to celestrak.org timed out. (connect timeout=10)'))
ERROR | https://celestrak.org/satcat/records.csv | HTTPSConnectionPool(host='celestrak.org', port=443): Max retries exceeded with url: /satcat/records.csv (Caused by ConnectTimeoutError(<HTTPSConnection(host='celestrak.org', port=443) at 0x24276ef8d90>, 'Connection to celestrak.org timed out. (connect timeout=10)'))


In [12]:
import pandas as pd

df = pd.read_excel('UCS-Satellite-Database 5-1-2023.xlsx')

# EO + LEO 필터링
eo_leo = df[
    df['Purpose'].str.contains('Earth Observation', na=False) &
    (df['Class of Orbit'] == 'LEO')
]

print(f"EO + LEO 위성 수: {len(eo_leo)}")
print(f"NORAD ID 있는 것: {eo_leo['NORAD Number'].notna().sum()}")

# NORAD ID 리스트 추출
norad_list = eo_leo['NORAD Number'].dropna().astype(int).tolist()
print(f"\n샘플 NORAD IDs: {norad_list[:10]}")

# 위성 이름 샘플
print("\n위성 이름 샘플:")
print(eo_leo[['Name of Satellite, Alternate Names', 'NORAD Number', 'Operator/Owner']].head(15).to_string())

EO + LEO 위성 수: 1192
NORAD ID 있는 것: 1192

샘플 NORAD IDs: [44859, 55107, 41460, 43600, 31304, 43768, 44103, 41785, 36798, 41786]

위성 이름 샘플:
                                             Name of Satellite, Alternate Names  NORAD Number                                            Operator/Owner
0                1HOPSAT-TD (1st-generation High Optical Performance Satellite)         44859                                              Hera Systems
1                                                       AAC AIS-Sat1 (Kelpie 1)         55107                                           AAC Clyde Space
3                                                                         AAt-4         41460                                     University of Aalborg
26                                                                       Aeolus         43600                               European Space Agency (ESA)
43                                          AIM (Aeronomy of Ice in Mesosphere)         31304  Center f